# STSELab research task — guided analysis

**Mostafizur Rahman Khan · 13 requirements · 100 repeated evaluations**

This notebook runs the same Python pipeline as `python run_analysis.py`. The ZIP includes precomputed outputs, so running it is optional if you only want to inspect them.

**Important:** counts describe what the LLM flagged. Proposed reason labels require your review. No expert defect truth or completed human judgments are included. Read `docs/START_HERE.md` and `docs/WORKED_EXAMPLE_PR1.md`.

Run cells in order. Google Colab needs internet to install packages. The analysis itself uses no API, GPU, model download, or external data service. Uploading your files to Colab places them in your Colab session; the offline review page can be used on your own computer.


## 1. Locate the project

**Local Jupyter:** open this notebook inside the unzipped project folder.

**Google Colab:** upload the complete delivered ZIP when prompted. The extraction step rejects unsafe archive paths. Keep the entire folder structure.


In [ ]:
from pathlib import Path
import sys, zipfile, subprocess, importlib.util

IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
ROOT = Path.cwd()
if not (ROOT / 'run_analysis.py').exists():
    candidate = ROOT / 'STSELab_Research_Package'
    if (candidate / 'run_analysis.py').exists():
        ROOT = candidate
    elif IN_COLAB:
        from google.colab import files
        print('Upload the complete STSELab_Research_Package.zip')
        uploads = files.upload()
        archives = [Path(name) for name in uploads if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one project ZIP.')
        destination = Path.cwd() / 'stselab_session'
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            for entry in z.infolist():
                target = (destination / entry.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe path inside ZIP: '+entry.filename)
            z.extractall(destination)
        scripts = list(destination.rglob('run_analysis.py'))
        if len(scripts) != 1:
            raise ValueError('Expected one run_analysis.py in the uploaded project ZIP.')
        ROOT = scripts[0].parent
    else:
        raise FileNotFoundError('Open this notebook inside the unzipped STSELab_Research_Package folder.')
ROOT = ROOT.resolve()
print('Project:', ROOT)
print('Python:', sys.version.split()[0])
assert (ROOT / 'inputs/raw_data.xlsx').exists()
ACTIVE_OUT = ROOT / 'outputs/draft'


## 2. Install the dependencies

This installs the listed compatible versions. For a fresh environment matching the delivered results, change the filename below to `requirements-tested.txt`. Python 3.11 or newer is required. If you already installed the packages locally, you may skip this cell.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)


## 3. Check the analysis, then run it

The tests check data validation, denominators, reproducibility, review provenance, and unresolved reason sets. The pipeline reads every worksheet and exports the complete opportunity grid, not only flagged items.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'run_analysis.py'], cwd=ROOT, check=True)
ACTIVE_OUT = ROOT / 'outputs/draft'


## 4. Inspect the numerical results

Each requirement–criterion pair has 100 opportunities. Overall criterion intervals resample whole runs across this fixed set of 13 requirements. They do not represent all engineering requirements or all LLMs.


In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
display(pd.read_csv(ACTIVE_OUT / 'tables/T03_flag_counts_matrix.csv'))
display(pd.read_csv(ACTIVE_OUT / 'tables/T05_criterion_summary.csv'))
display(pd.read_csv(ACTIVE_OUT / 'tables/T10_review_coverage.csv'))


## 5. View the scientific figures

All six figures are exported as 300-dpi PNG plus vector PDF and SVG. The reason figures are provisional until labels have been reviewed. The evidence/criterion-fit figure initially displays pending judgments; that is intentional.


In [ ]:
for image_path in sorted((ACTIVE_OUT / 'figures').glob('*.png')):
    display(Markdown('### '+image_path.stem))
    display(Image(filename=str(image_path), width=1000))


## 6. Read the report and review explanations

The report and table book are in the output folder. Download the project with the final cell, then open `outputs/draft/Report_Draft.html` and `outputs/draft/review.html` on your own computer.

In the offline review page:
1. Read the exact explanation, requirement, and criterion definition.
2. Check every proposed reason. Preserve secondary reasons; add a category when needed.
3. Mark labels **Reviewed by me** only after checking them.
4. Assess source evidence and criterion fit separately, with your name, source location, and reasoning.
5. Save, then export `annotations_reviewed.csv`. Store it in `working/`.

Start with `data/review_queue.csv`, including the eight initially unmatched cases, but review the matched cases too. The queue is a prioritization aid, not a random validation sample. Changes to the codebook belong in `config/reason_codebook.json`; the CSV copy is for reading.


In [ ]:
display(Markdown((ACTIVE_OUT / 'Report_Draft.md').read_text(encoding='utf-8')))


## 7. Optional — import your completed or partial review

Run this cell **after** exporting the review CSV. On Colab it asks you to upload that file; locally place it in `working/annotations_reviewed.csv`. It preserves earlier draft outputs and writes a reviewed draft separately. Unreviewed labels and pending judgments remain visible.


In [ ]:
review_path = ROOT / 'working/annotations_reviewed.csv'
if IN_COLAB:
    from google.colab import files
    print('Upload annotations_reviewed.csv exported from the offline review page')
    uploads = files.upload()
    csv_files = [Path(name) for name in uploads if name.lower().endswith('.csv')]
    if len(csv_files) != 1:
        raise ValueError('Upload exactly one review CSV.')
    review_path.write_bytes(csv_files[0].read_bytes())
if review_path.exists():
    subprocess.run([sys.executable, 'run_analysis.py', '--reviewed', str(review_path),
                    '--out', str(ROOT / 'outputs/reviewed_draft')], cwd=ROOT, check=True)
    ACTIVE_OUT = ROOT / 'outputs/reviewed_draft'
    print('Reviewed draft generated. Rerun cells 4–6 to inspect it.')
else:
    print('No review CSV found; keep using the draft and return to this step later.')


## 8. Final mode — only after completing the research review

Final mode requires reviewed reasons, completed evidence/fit judgments, and accepted definitions for every used code. It checks recorded completion; it cannot certify scientific correctness or reviewer expertise. Honest `not_verifiable` and `not_judgable` judgments are allowed when explained.

Use this command locally, or with `subprocess.run` in another cell:

```bash
python run_analysis.py --reviewed working/annotations_reviewed.csv --mode final --out outputs/final
```

Revise the report and presentation using your actual findings, limitations, and review procedure. Do not describe initial regex proposals as human-validated conclusions.


## 9. Download your work

This archives the full project, including input data, code, configuration, exported review files, and generated outputs. Colab sessions are temporary: download before ending the session. The ZIP is created outside the project, so it does not include itself.


In [ ]:
archive_path = ROOT.parent / 'STSELab_Research_Results.zip'
excluded_parts = {'.venv', 'venv', '__pycache__', '.ipynb_checkpoints', '.git'}
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(ROOT.rglob('*')):
        rel = path.relative_to(ROOT)
        if path.is_file() and not excluded_parts.intersection(rel.parts) and path.suffix != '.pyc':
            z.write(path, Path(ROOT.name) / rel)
print('Saved:', archive_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))
else:
    from IPython.display import FileLink
    print('Open the saved path above in your file browser.')
